# Engine validation — Monte Carlo (no data required)

Under the null hypothesis (all alphas equal 0) with normal residuals, generating synthetic
data thousands of times means the GRS statistic should exactly follow $F(N,\,T-N-K)$ in
finite samples. If the 5% rejection rate comes out near ~5% and the distribution matches,
then **the formula and the degrees of freedom are validated simultaneously**.

> Run from the `tests/` folder — validation calls `grs_test` **directly from the real engine (`ff_core.py`)**, i.e. the actual code, not a copy.

In [1]:
import sys; sys.path.insert(0, '../src')    # load src/ff_core.py
import numpy as np
from scipy import stats
from ff_core import grs_test                     # GRS from the real engine
RNG = np.random.default_rng(20260628)

def fast_alpha_resid(Y, F):
    X = np.column_stack([np.ones(len(Y)), F])
    coef = np.linalg.solve(X.T @ X, X.T @ Y)
    return coef[0], Y - X @ coef

def simulate(T, N, K, reps, alpha_inject=0.0):
    rej = 0; Fs = np.empty(reps)
    scale = np.array([1.0] + [0.6]*(K-1)); mean = np.array([0.5] + [0.25]*(K-1))
    for r in range(reps):
        F = RNG.normal(size=(T, K)) * scale + mean
        B = RNG.uniform(0.5, 1.5, size=(N, K)); E = RNG.normal(size=(T, N))
        Y = np.full(N, alpha_inject) + F @ B.T + E
        a, res = fast_alpha_resid(Y, F)
        out = grs_test(a, res, F)                 # call the engine function
        Fs[r] = out['F']
        if out['p_value'] < 0.05: rej += 1
    return rej / reps, Fs

## Size test — is the rejection rate under the null equal to the nominal level?

In [2]:
T, N, K, reps = 720, 25, 3, 5000
size, Fs = simulate(T, N, K, reps, 0.0)
ks = stats.kstest(Fs, lambda x: stats.f.cdf(x, N, T - N - K))
print(f"5% rejection rate : {size:.4f}   (target 0.0500)")
print(f"statistic mean    : {Fs.mean():.4f}   (theory {(T-N-K)/(T-N-K-2):.4f})")
print(f"KS vs F dist      : p = {ks.pvalue:.4f}   (large = distribution matches)")
print('verdict:', 'engine validated — formula and degrees of freedom correct'
      if abs(size - 0.05) < 0.01 and ks.pvalue > 0.05 else 'needs recheck')

5% 기각률 : 0.0500   (목표 0.0500)
통계량 평균: 1.0010   (이론 1.0029)
KS vs F분포: p = 0.8707   (크면 분포 일치)
판정: 엔진 검증 통과 — 수식·자유도 정확


## Power — does injecting alpha raise the rejection rate?

In [3]:
for a in (0.05, 0.10):
    p5, _ = simulate(T, N, K, 1500, a)
    print(f"alpha = {a:.2f}  →  5% rejection rate {p5:.4f}")

alpha = 0.05  →  5% 기각률 0.8913


alpha = 0.10  →  5% 기각률 1.0000
